# ChildHealth Explorer - raw data exploration

Lab 2, Task 2.1. We retain the approved study: 20 countries, 2000-2023, both sexes. The mandatory WHO, World Bank and OWID integrations supplement UNICEF vaccination data.

This notebook downloads missing snapshots and transforms saved files. Existing snapshots are verified and reused; run `python -m etl.extract --refresh` deliberately to update them. No database writes occur here. Raw files remain outside Git; these outputs record the inspected snapshot.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "etl").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
from etl.extract import download_indicator
from etl.study import indicators, study
from etl.transform import transform_indicator
config = study()
frames, profiles = {}, []
print("Study:", config["start_year"], "to", config["end_year"], "|", len(config["pilot_countries"]), "countries")
print("Kernel:", sys.executable)

Study: 2000 to 2023 | 20 countries
Kernel: C:\Users\Andrei Bujor\Documents\ChatGPT\aaaaaa\.venv\Scripts\python.exe


## Inspect each saved source

`head()`, `info()` and `isna().sum()` are shown for every source-specific indicator. The missing-value percentage is reported for both all normalized downloaded rows and the study subset. Missing country-year combinations absent from the response are counted separately from explicit null values. WHO non-national age/wealth dimensions excluded by transformation are reported separately, rather than being mistaken for cleaning rejections.

In [2]:
for item in indicators():
    path, url = download_indicator(item)
    df = transform_indicator(item)
    key = f"{item['source']}/{item['code']}"
    frames[key] = df
    print("\n" + key, "\nRequest:", url, "\nSnapshot:", path.relative_to(ROOT))
    display(df.head())
    df.info()
    display(df.isna().sum().rename("missing_values"))
    years = pd.to_numeric(df["year"], errors="coerce")
    subset = df.loc[df["country_iso3"].isin(config["pilot_countries"]) & years.between(config["start_year"], config["end_year"])]
    known = subset.loc[subset["value"].notna(), ["country_iso3", "year"]].drop_duplicates()
    expected = len(config["pilot_countries"]) * (config["end_year"] - config["start_year"] + 1)
    profiles.append({"source":item["source"], "indicator":item["code"], "normalized_rows":len(df),
                     "missing_values":int(df["value"].isna().sum()),
                     "missing_percent":round(float(df["value"].isna().mean()*100), 3),
                     "study_rows":len(subset), "study_missing_values":int(subset["value"].isna().sum()),
                     "study_missing_percent":round(float(subset["value"].isna().mean()*100), 3),
                     "study_country_year_gaps":expected-len(known),
                     "excluded_dimensions":df.attrs.get("excluded_dimensions",0)})


unicef/IM_DTP3 
Request: https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,GLOBAL_DATAFLOW,1.0/ROU+BGR+DEU+FRA+POL+ESP+GBR+SWE+USA+CAN+BRA+MEX+IND+IDN+ZAF+KEN+NGA+EGY+AUS+JPN.IM_DTP3._T?format=sdmx-json&startPeriod=2000&endPeriod=2023 
Snapshot: data\raw\unicef\IM_DTP3_2000_2023.json


,country_iso3,indicator_code,year,sex,value
0,MEX,IM_DTP3,2000,BTSX,97
1,MEX,IM_DTP3,2001,BTSX,97
2,MEX,IM_DTP3,2002,BTSX,97
3,MEX,IM_DTP3,2003,BTSX,98
4,MEX,IM_DTP3,2004,BTSX,98


<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   country_iso3    480 non-null    str   
 1   indicator_code  480 non-null    str   
 2   year            480 non-null    int64 
 3   sex             480 non-null    str   
 4   value           480 non-null    object
dtypes: int64(1), object(1), str(3)
memory usage: 18.9+ KB


country_iso3      0
indicator_code    0
year              0
sex               0
value             0
Name: missing_values, dtype: int64


unicef/IM_MCV1 
Request: https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,GLOBAL_DATAFLOW,1.0/ROU+BGR+DEU+FRA+POL+ESP+GBR+SWE+USA+CAN+BRA+MEX+IND+IDN+ZAF+KEN+NGA+EGY+AUS+JPN.IM_MCV1._T?format=sdmx-json&startPeriod=2000&endPeriod=2023 
Snapshot: data\raw\unicef\IM_MCV1_2000_2023.json


,country_iso3,indicator_code,year,sex,value
0,MEX,IM_MCV1,2000,BTSX,96
1,MEX,IM_MCV1,2001,BTSX,95
2,MEX,IM_MCV1,2002,BTSX,96
3,MEX,IM_MCV1,2003,BTSX,96
4,MEX,IM_MCV1,2004,BTSX,96


... (26 outputs left out)


owid/child-mortality 
Request: https://ourworldindata.org/grapher/child-mortality.csv 
Snapshot: data\raw\owid\child-mortality_2000_2023.csv


,country_iso3,indicator_code,year,sex,value
0,AFG,child-mortality,1957,BTSX,368.2
1,AFG,child-mortality,1958,BTSX,362.1
2,AFG,child-mortality,1959,BTSX,356.4
3,AFG,child-mortality,1960,BTSX,350.6
4,AFG,child-mortality,1961,BTSX,345.2


<class 'pandas.DataFrame'>
RangeIndex: 17066 entries, 0 to 17065
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    17066 non-null  str    
 1   indicator_code  17066 non-null  str    
 2   year            17066 non-null  int64  
 3   sex             17066 non-null  str    
 4   value           17066 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 666.8 KB


country_iso3      0
indicator_code    0
year              0
sex               0
value             0
Name: missing_values, dtype: int64

## Missingness summary

Explicit missing measurements will be rejected during cleaning, never converted to zero. Country-year gaps describe coverage; an absent observation is not an imported or rejected database row. Complete chart downloads include countries and years outside the pilot, which later scope rules remove.

In [3]:
summary = pd.DataFrame(profiles)
display(summary)
output = ROOT / "docs/evidence/lab2-exploration.json"
output.write_text(json.dumps(profiles, indent=2) + "\n", encoding="utf-8")
print("Saved:", output.relative_to(ROOT))

,source,indicator,normalized_rows,missing_values,missing_percent,study_rows,study_missing_values,study_missing_percent,study_country_year_gaps,excluded_dimensions
0,unicef,IM_DTP3,480,0,0.000,480,0,0.000,0,0
1,unicef,IM_MCV1,480,0,0.000,480,0,0.000,0,0
2,worldbank,SH.DYN.MORT,6384,528,8.271,480,0,0.000,0,0
3,worldbank,NY.GDP.PCAP.PP.KD,6384,511,8.004,480,0,0.000,0,0
4,worldbank,SH.XPD.CHEX.GD.ZS,6384,692,10.840,480,0,0.000,0,0
5,worldbank,SH.H2O.BASW.ZS,6384,236,3.697,480,10,2.083,10,0
6,worldbank,SH.STA.BASS.ZS,6384,262,4.104,480,1,0.208,1,0
7,who,MDG_0000000007,4800,0,0.000,480,0,0.000,0,12720
8,owid,child-mortality,17066,0,0.000,480,0,0.000,0,0


Saved: docs\evidence\lab2-exploration.json


## Interpretation

The three mortality series remain distinct by source. OWID mortality values have been converted from deaths per 100 births to per 1,000 using the saved unit metadata. Shared UN IGME provenance means these series cannot be treated as independent training observations. Pandas object/string dtypes in uncleaned data are expected: numeric conversion, identifier harmonization and rejection rules belong to the next step.